In [1]:
import os
os.chdir('/home/elious/research_projects/mdpi_sensors_2026')

# Notebook 06: Statistical Tests

1. McNemar test , 6 pairs per dataset (C(4,2))
2. RF 5-seed robustness (seeds 42-46)

In [2]:
import numpy as np
import pandas as pd
from itertools import combinations
from statsmodels.stats.contingency_tables import mcnemar
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score
import json, time, warnings
warnings.filterwarnings('ignore')
print('Libraries loaded.')

Libraries loaded.


## 1. McNemar Tests

In [3]:
MODELS = ['RandomForest', 'DecisionTree', 'XGBoost', 'LogisticRegression']
ALPHA  = 0.05
results_mc = []

for dataset, prefix in [('UGRansome2024', 'ugr'), ('CICIoT2023', 'cic')]:
    preds = {}
    for m in MODELS:
        df = pd.read_csv(f'results/baselines/{prefix}_{m}_predictions.csv')
        preds[m] = df

    y_true = preds[MODELS[0]]['y_true'].values

    for m1, m2 in combinations(MODELS, 2):
        p1 = preds[m1]['y_pred'].values
        p2 = preds[m2]['y_pred'].values

        b = int(np.sum((p1 == y_true) & (p2 != y_true)))  # A right, B wrong
        c = int(np.sum((p1 != y_true) & (p2 == y_true)))  # A wrong, B right
        # McNemar table [[n11, b],[c, n22]] , only b, c matter; diagonal set to 0
        res = mcnemar([[0, b], [c, 0]], exact=False, correction=True)
        sig = res.pvalue < ALPHA
        results_mc.append({
            'dataset': dataset,
            'model_A': m1,
            'model_B': m2,
            'b': b,
            'c': c,
            'statistic': round(res.statistic, 4),
            'p_value': round(res.pvalue, 6),
            'significant': sig
        })
        print(f'{dataset} | {m1} vs {m2}: stat={res.statistic:.4f}, p={res.pvalue:.6f}, sig={sig}')

df_mc = pd.DataFrame(results_mc)
df_mc.to_csv('results/mcnemar_results.csv', index=False)
print('\nSaved results/mcnemar_results.csv')
df_mc

UGRansome2024 | RandomForest vs DecisionTree: stat=12.1212, p=0.000499, sig=True
UGRansome2024 | RandomForest vs XGBoost: stat=23.3472, p=0.000001, sig=True
UGRansome2024 | RandomForest vs LogisticRegression: stat=1162.8695, p=0.000000, sig=True
UGRansome2024 | DecisionTree vs XGBoost: stat=6.5574, p=0.010445, sig=True
UGRansome2024 | DecisionTree vs LogisticRegression: stat=1204.2493, p=0.000000, sig=True
UGRansome2024 | XGBoost vs LogisticRegression: stat=1272.2424, p=0.000000, sig=True
CICIoT2023 | RandomForest vs DecisionTree: stat=12.9933, p=0.000313, sig=True
CICIoT2023 | RandomForest vs XGBoost: stat=28.6284, p=0.000000, sig=True
CICIoT2023 | RandomForest vs LogisticRegression: stat=622.4615, p=0.000000, sig=True
CICIoT2023 | DecisionTree vs XGBoost: stat=4.5333, p=0.033241, sig=True
CICIoT2023 | DecisionTree vs LogisticRegression: stat=514.4379, p=0.000000, sig=True
CICIoT2023 | XGBoost vs LogisticRegression: stat=552.8968, p=0.000000, sig=True

Saved results/mcnemar_results.cs

,dataset,model_A,model_B,b,c,statistic,p_value,significant
0,UGRansome2024,RandomForest,DecisionTree,6,27,12.1212,0.000499,True
1,UGRansome2024,RandomForest,XGBoost,15,57,23.3472,0.000001,True
2,UGRansome2024,RandomForest,LogisticRegression,1329,58,1162.8695,0.000000,True
3,UGRansome2024,DecisionTree,XGBoost,20,41,6.5574,0.010445,True
4,UGRansome2024,DecisionTree,LogisticRegression,1338,46,1204.2493,0.000000,True
5,UGRansome2024,XGBoost,LogisticRegression,1333,20,1272.2424,0.000000,True
6,CICIoT2023,RandomForest,DecisionTree,97,52,12.9933,0.000313,True
7,CICIoT2023,RandomForest,XGBoost,149,69,28.6284,0.000000,True
8,CICIoT2023,RandomForest,LogisticRegression,767,52,622.4615,0.000000,True
9,CICIoT2023,DecisionTree,XGBoost,145,110,4.5333,0.033241,True


## 2. RF 5-Seed Robustness

In [4]:
SEEDS = [42, 43, 44, 45, 46]
results_seed = []

for dataset, train_path, test_path, label_col in [
    ('UGRansome2024', 'data/processed/ugr_train.csv', 'data/processed/ugr_test.csv', 'Prediction'),
    ('CICIoT2023',    'data/processed/cic_train.csv', 'data/processed/cic_test.csv',  'label_binary'),
]:
    print(f'\n{dataset} , loading ...')
    train_df = pd.read_csv(train_path)
    test_df  = pd.read_csv(test_path)

    if dataset == 'CICIoT2023':
        drop_cols = [c for c in ['label', 'label_binary'] if c in train_df.columns]
        X_tr = train_df.drop(columns=drop_cols)
        y_tr = train_df['label_binary']
        X_te = test_df.drop(columns=['label', 'label_binary'])
        y_te = test_df['label_binary']
    else:
        X_tr = train_df.drop(columns=['Prediction'])
        y_tr = train_df['Prediction']
        X_te = test_df.drop(columns=['Prediction'])
        y_te = test_df['Prediction']

    for seed in SEEDS:
        t0 = time.perf_counter()
        rf = RandomForestClassifier(n_estimators=100, random_state=seed, class_weight='balanced')
        rf.fit(X_tr, y_tr)
        y_pred = rf.predict(X_te)
        f1 = f1_score(y_te, y_pred, average='macro')
        elapsed = time.perf_counter() - t0
        results_seed.append({'dataset': dataset, 'seed': seed, 'f1_macro': round(f1, 6), 'time_sec': round(elapsed, 3)})
        print(f'  seed={seed}: F1={f1:.6f}')

df_seed = pd.DataFrame(results_seed)
df_seed.to_csv('results/rf_seed_robustness.csv', index=False)
print('\nSaved results/rf_seed_robustness.csv')

print('\nSummary (mean ± std):')
print(df_seed.groupby('dataset')['f1_macro'].agg(['mean','std']).round(6))


UGRansome2024 , loading ...


  seed=42: F1=0.991080


  seed=43: F1=0.990764


  seed=44: F1=0.991000


  seed=45: F1=0.991318


...

  seed=42: F1=0.962162


  seed=43: F1=0.959774


  seed=44: F1=0.957806


  seed=45: F1=0.960902


  seed=46: F1=0.960780

Saved results/rf_seed_robustness.csv

Summary (mean ± std):
                   mean       std
dataset                          
CICIoT2023     0.960285  0.001624
UGRansome2024  0.991001  0.000217


In [5]:
print('Notebook 06 complete.')

Notebook 06 complete.
